In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import patsy
from scipy import stats
import warnings
warnings.simplefilter("ignore", FutureWarning)
METABOTKIT = r'C:\Users\jaileru\Projects\metabolomics-toolkit'
ROOT = r"\\172.22.48.112\Sumner_HHEAR_Lichtveld\4a. Data_LC-MS_OrbiTrap\5-Statistical Analyses\Julian Analysis\July-2026"
import sys,os
sys.path.append(os.path.abspath(METABOTKIT))
from differential_abundance import DifferentialAbundance
from dimensionality_reduction import pca_plot, OPLSDA
PROJECT_ID = 'ST002118'
from itertools import chain
from docx import Document
sys.path.append(os.path.abspath(os.path.join(ROOT, "Analysis")))
from ST002118 import *

In [13]:
# rpy2 shells out to `R CMD config`/`R RHOME` via cmd.exe, which cannot start
# with a UNC path as its working directory (it silently falls back to another
# directory and the R call's output comes back empty, crashing rpy2's parser
# with IndexError). Switch to a local directory before importing rpy2 so the
# subprocess calls it makes don't inherit a UNC cwd. All other paths in this
# notebook are already absolute (via ROOT), so this is safe.
os.chdir(os.path.expanduser('~'))

In [14]:
# On this machine, `R CMD config --ldflags` fails to find `make` inside R's
# internal shell (make.exe is installed but not resolvable from R's msys2
# subshell), so it returns an empty, exit-code-0 result instead of raising an
# error. rpy2 has a built-in fallback for "Rtools not installed" that adds the
# R DLL directory directly without needing R CMD config at all, but it only
# triggers on subprocess.CalledProcessError, not on this empty-output case,
# so it crashes with IndexError instead. Patch it to treat empty output the
# same way, so rpy2's own fallback kicks in. Must run before rpy2.robjects
# (or anything importing rpy2.rinterface) is imported anywhere in this kernel.
import rpy2.situation

_orig_get_r_cmd_config = rpy2.situation._get_r_cmd_config

def _patched_get_r_cmd_config(r_home, about, allow_empty=False):
    try:
        return _orig_get_r_cmd_config(r_home, about, allow_empty=allow_empty)
    except IndexError:
        raise rpy2.situation.subprocess.CalledProcessError(1, ('R', 'CMD', 'config', about))

rpy2.situation._get_r_cmd_config = _patched_get_r_cmd_config

In [2]:
untargeted_data = clean_untargeted_reference(rf'{ROOT}/Data/Untargeted/reference_copy_untargeted_data.parquet')
categorical_data = clean_categorical_reference(rf'{ROOT}/Data/Untargeted/reference_copy_categorical_data.parquet',
                                               additional_cols=['v113a','v113b','v113c','v113d','v108a','v107g'])
targeted_data = pd.read_excel(r"\\172.22.48.112\Sumner_HHEAR_Lichtveld\4a. Data_LC-MS_OrbiTrap\5-Statistical Analyses\Julian Analysis\July-2026\Data\Targeted\TARGETED_DATA_WIDE_07132026.xlsx")
targeted_data.drop(columns='CompositeKey',inplace=True)
targeted_data.set_index("SID_MW",inplace=True)
categorical_data = categorical_data[categorical_data['Trimester_n'] == 1]
categorical_data = categorical_data.loc[targeted_data.index.intersection(categorical_data.index),:]

Cleaning Reference Dataset
Saving additional columns ['v113a', 'v113b', 'v113c', 'v113d', 'v108a', 'v107g']
Renaming Columns
Renaming Values


In [3]:
# Build Hg category only for rows where Hg was actually measured
hg_data = targeted_data[['Hg (ng/mL)']].copy()
hg_data['Hg (ng/mL)'] = hg_data['Hg (ng/mL)'].astype(float)

hg_data['Hg EPA'] = np.where(hg_data['Hg (ng/mL)'] < 5.8, 'normal', 'high')

# Merge into t1_categorical on subject ID, leaving non-matches as NaN
categorical_data = categorical_data.merge(
    hg_data,
    left_index=True,
    right_index=True
)
categorical_data.head()

,StatusAtBirth,GenderBaby,Birthweight(grams),BirthweightLessThan2500g,GestationalAgeAtIntake,PretermBirthLessThan37weeks,APGARScoreAt5min,APGARScoreLessThan7At5min,AdverseBirthOutcome,Trimester_n,Ethnicity9Categories,Ethnicity8Categories,Ethnicity7Categories,Ethnicity6Categories,DistrictAtIntake10Categories,MaternalAgeAtIntake3Categories,Hg (ng/mL),Hg EPA
Lich_S_69,2.0,1.0,3170.0,no,41.0,no,9.0,2.0,2.0,1.0,2.0,2.0,1.0,Creole,Paramaribo,20-34 years,5.76,normal
Lich_S_449,2.0,1.0,3305.0,no,40.2,no,10.0,2.0,2.0,1.0,3.0,3.0,2.0,Hindustani,Nickerie,20-34 years,4.42,normal
Lich_S_627,2.0,1.0,2255.0,yes,37.5,no,10.0,2.0,1.0,1.0,9.0,8.0,6.0,Mixed,Wanica,35+ years,1.05,normal
Lich_S_643,2.0,1.0,2860.0,no,37.0,no,10.0,2.0,2.0,1.0,5.0,5.0,4.0,Javanese,Nickerie,20-34 years,3.12,normal
Lich_S_428,2.0,2.0,3180.0,no,37.3,no,10.0,2.0,2.0,1.0,9.0,8.0,6.0,Mixed,Nickerie,35+ years,2.27,normal


In [4]:
categorical_data['Hg Tertiles'] = pd.qcut(hg_data['Hg (ng/mL)'],q=4,labels=['T1','T2','T3','T4'])

In [5]:
pca_plot(targeted_data.dropna(),categorical_data,applylog=True,hue=['Hg EPA','Hg Tertiles','MaternalAgeAtIntake3Categories','Ethnicity6Categories','BirthweightLessThan2500g','PretermBirthLessThan37weeks','DistrictAtIntake10Categories'],backend='plotly')

,PC1,PC2,DistrictAtIntake10Categories
SID_MW,,,
Lich_S_69,0.880114,-1.881708,Paramaribo
Lich_S_449,0.109524,0.659049,Nickerie
Lich_S_627,-0.571066,-2.118468,Wanica
Lich_S_643,0.011686,-0.598363,Nickerie
Lich_S_428,-0.961218,1.356666,Nickerie
...,...,...,...
Lich_S_328,-0.863817,-1.385201,Para
Lich_S_216,-0.431639,-1.343186,Paramaribo
Lich_S_383,1.965758,1.377433,Wanica
